## ETL/Gold/02 - dim_modelo_scd2 (SCD Type 2)
## Implementado con spark.sql:
##   PASO 1: cerrar version vigente si el atributo cambia
##   PASO 2: insertar nueva version vigente

In [0]:
%py

def get_param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG = get_param("catalog", "pf")
S_MODELO = f"{CATALOG}.silver.modelos"
DIM      = f"{CATALOG}.gold.dim_modelo_scd2"

In [0]:
%py

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW staging_modelos AS
    SELECT
        model_id,
        nombre,
        org_id,
        pipeline_tag,
        library_name,
        COALESCE(license_tag, 'sin_licencia') AS license_tag
    FROM {S_MODELO}
    WHERE ingestion_date = (SELECT MAX(ingestion_date) FROM {S_MODELO})
""")

In [0]:
%py

n_cerradas = spark.sql(f"""
    MERGE INTO {DIM} AS t
    USING staging_modelos AS s
    ON  t.model_id = s.model_id AND t.is_current = TRUE
    WHEN MATCHED AND (
            t.nombre IS DISTINCT FROM s.nombre OR
            t.org_id IS DISTINCT FROM s.org_id OR
            t.pipeline_tag IS DISTINCT FROM s.pipeline_tag OR
            t.library_name IS DISTINCT FROM s.library_name OR
            t.license_tag IS DISTINCT FROM s.license_tag
         )
    THEN UPDATE SET t.valid_to = CURRENT_TIMESTAMP(), t.is_current = FALSE
""")

#print(f"Versiones cerradas en esta corrida: {n_cerradas.numUpdatedRows}")
historial = spark.sql(f"""
    DESCRIBE HISTORY {DIM}
""")

ultima_operacion = historial.orderBy("version", ascending=False).first()

#print(ultima_operacion.operationMetrics)
metricas = ultima_operacion.operationMetrics

n_cerradas = int(metricas.get("numTargetRowsUpdated", 0))

print(f"Versiones cerradas en esta corrida: {n_cerradas}")

In [0]:
%py

n_insert = spark.sql(f"""
    INSERT INTO {DIM} (model_id, nombre, org_id, pipeline_tag, library_name,
                       license_tag, valid_from, valid_to, is_current, _createdAt)
    SELECT s.model_id, s.nombre, s.org_id, s.pipeline_tag, s.library_name, s.license_tag,
           CURRENT_TIMESTAMP(), '9999-12-31', TRUE, CURRENT_TIMESTAMP()
    FROM staging_modelos s
    WHERE NOT EXISTS (
        SELECT 1 FROM {DIM} t
        WHERE t.model_id = s.model_id AND t.is_current = TRUE
    )
""")

print(f"Filas insertadas como nuevas versiones: {n_insert}")

In [0]:
%py

spark.sql(f"""
    SELECT is_current, COUNT(*) AS n
    FROM {DIM}
    GROUP BY is_current
""").show()

In [0]:
%py

spark.sql(f"""
    SELECT COUNT(*) AS modelos_sin_una_vigente
    FROM (
        SELECT model_id
        FROM {DIM}
        WHERE is_current = TRUE
        GROUP BY model_id
        HAVING COUNT(*) <> 1
    )
""").show()